# Scientific Programming: A Crash Course

## Unit 3 – Data Management

So far we've learned all the core structures of programming: the basic data types (ints, lists, strings, etc.), the fundamental control structures (for-loops, while-loops, if-statements), and higher level abstractions (functions and objects). This is a powerful set of tools that will let you accomplish a lot of stuff. Even if you decide not to stick with Python, the core principles we've covered so far are pretty universal, and they should prove useful wherever your coding journey takes you.

Before we put these tools to practical scientific uses, there's still one more foundational topic that we need to cover: data. Usually, your program will need to create, read, modify, and delete some kind of data stored permanently on your computer in separate files. Today we'll look at some commonly used data formats, how to read and write data, and how to reference other files on your computer. Then, I'll cover a few other more general topics about Python that are not super important, but still useful to know.

## Reading and Writing Files

So far, everything we've been doing has been very self-contained inside the notebook. But, in the real world, we very often need to interact with other files, either by reading data in or writing data out. Let's first look at how to read a file in Python:

In [ ]:
with open('some_file.txt') as file_object:
    content_of_the_file = file_object.read()
    
print(content_of_the_file)

When you run this code, you will get a `FileNotFoundError` error. Obviously this is because there is no file called `some_file.txt` on your computer right now. So, first, I want you to go and create a plain text file (it should have the file name extension `.txt`) and save it to the same folder as this notebook. Write a message inside the file like, `Ciao Mondo!`. You will need to use a text editor app like TextEdit (Mac) or Notepad (Windows), or some other text editor that you like. Once you've created the file, try running the code again and check that you can recover the message that you saved in the file.

Now let's look at the new syntax. Here we are using a **context manager** – the `with` statement. Context managers are not used very often (opening files is the most common use-case), so if you don't really get it, don't worry... just learn the syntax above so that you know how to read and write files.

To understand what's going on, we need to rewind back to the old days of Python, when you would open a file like this:

```python
file_object = open('some_file.txt')
content_of_the_file = file_object.read()
file_object.close()
```

The first line, where we create the file object, essentially initializes a connection to the file. In the second line, we use the file object's `.read()` method to read the content of the file, which we then assign to the variable `content_of_the_file`. Finally, we need to explicitly close the connection to the file by calling the file object's `.close()` method. Closing a file connection can be important because this releases Python's lock on the file, so that some other program can access it.

The context manager syntax simplifies this a little:

```python
with open('some_file.txt') as file_object:
    content_of_the_file = file_object.read()
```

As you can see, we are using the same bits of code – the `open()` function and the `.read()` method – and we have the same two variables – `file_object` and `content_of_the_file`. However, the context manager automates the process of closing the file connection. Any code indented inside the with-statement is run while the file connection is open. Once you exit the with-statement, the connection is automatically closed.

Now let's try writing a file:

In [ ]:
secret_message = 'Python is cool 😎'

with open('some_file.txt', 'w') as file:
    file.write(secret_message)

As, you can see the syntax is pretty similar. The only differences are (1) we need an extra `'w'` in the call to the `open()` function, which requests that the file be opened with write permissions, and (2) we need to use the file object's `.write()` method. Go ahead and verify that the message was correctly stored by opening `some_file.txt` in another app. Note, also, that the original content of the file is completely overwritten. Be careful when working with files: Python will quite happily overwrite data without any warnings. In fact, even just opening a connection to the file in `'w'` mode will erase the content of the file, even if you don't actually write anything:

In [ ]:
with open('some_file.txt', 'w') as file:
    pass

A useful feature of the file object is that it is iterable. When you iterate over a file object, you get access to each line of the file. This could be useful, for example, if you had a file that contains some data on each line and you wanted to read the file line by line. Before running the following code, edit your `some_file.txt` so that it contains multiple lines of stuff.

In [ ]:
with open('some_file.txt') as file:
    for line in file:
        print(line)

In [ ]:
######################### QUESTION 1 #########################
import quiz; quiz.display(quiz.HTML(quiz._unit3_questions[0]))
##############################################################

## Structured Data Formats

Storing data in plain text – as we just did above – is a great idea. Plain text is lightweight, accessible, platform agnostic, and unencumbered by any kind of proprietary lock-in. A plain text file made 50 years ago can still be opened today, and will – hopefully – still be openable in another 50 years' time. I highly encourage you to stop using proprietary, closed formats wherever you can, especially in science, where we want to maximize the accessibility, transparency, and longevity of our datasets.

Typically, our datasets are highly structured – for example, your data may be naturally arranged in a table or in a hierarchical structure. This means that storing such data as plain text can get a bit messy. For example, let's say you were running an experiment, and your script writes the data to plain text. You might decide that the first line of the file is the subject ID, the second line is the condition ID, then the next *n* lines contain practice trials, and the next *m* lines contain real trials, and the last line records the time the experiment ended. Each of the lines that records an individual trial might contain two pieces of information – the test stimulus and the participant's response, separated by a dash. This is messy, right? Anyone looking at this file will not be able to easily interpret it, and it's also difficult for you yourself to work with computationally.

Here's my advice: Don't try to reinvent the wheel by inventing your own ad-hoc plain text data format; instead, use a standard format to store your data. The two most popular are CSV and JSON. You've probably encountered CSV files before, but maybe not JSON. Both are useful, but they have different philosophies and ideal use-cases. Crucially, both formats are actually just extensions of plain text. CSV and JSON files are simply plain text files that follow certain conventions to indicate their own structure. This means we get the benefits of plain text, as well as the benefits of something more structured.

### CSV files

The CSV (comma-separated values) format mirrors the basic structure of a spreadsheet with rows and columns. For example, you might have a CSV file in which the raw plain text looks like this:

```
subject,test_type,system,correct
1,production,size,1
1,production,size,1
1,production,size,0
1,production,size,1
1,production,size,1
...
240,comprehension,shape,0
240,comprehension,shape,1
240,comprehension,shape,1
240,comprehension,shape,0
240,comprehension,shape,0
```

And here's what it looks like if I tidy up the alignment a little so that you can read it better:

```
subject, test_type,     system, correct
1,       production,    size,   1
1,       production,    size,   1
1,       production,    size,   0
1,       production,    size,   1
1,       production,    size,   1
...
240,     comprehension, shape,  0
240,     comprehension, shape,  1
240,     comprehension, shape,  1
240,     comprehension, shape,  0
240,     comprehension, shape,  0
```

The details of the dataset don't matter; the main point is we have a tabular structure. Each row is on a new line and each cell within a row is separated by a comma. Deciding how to organize your data in a table would require an entire class in itself, so I won't go into too much detail here; instead, I'd recommend you read up on the "Tidy Data" philosophy here: https://tidyr.tidyverse.org/articles/tidy-data.html (perhaps bookmark it for the weekend – the code is in R, but the advice is universal).

Anyhow, we could try to parse this CSV file manually. For example, we could iterate over the lines in the file and split each line into parts based on the comma character. If you're feeling adventurous, try writing a function to parse a CSV file; hint: strings have a `.split()` method that should be useful:

In [ ]:
def my_csv_reader(file_name):
    # write your function here
    return

However, there are already functions available to you for reading CSV files; better to use something that is well-tested than to roll your own!

Let's import the `csv` module from the Python standard library:

In [ ]:
import csv

To use the module, you are expected to open the file yourself and use the relevant function to parse the CSV. This will result in code like this:

In [ ]:
with open('data/example_data.csv') as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)

This is somewhat helpful – at least it does the parsing for us – but still not much better than just manually iterating over the lines of the raw file. A better option is `DictReader()` where each row will be represented as a dictionary like this:

```python
{'subject': '1', 'test_type': 'production', 'category_system': 'size', 'correct': '1'}
```

which can make it easier to pull out the info you need. Have a look at the following code and try to understand what it does before you run it.

In [ ]:
n_correct = 0
total_trials = 0

with open('data/example_data.csv', newline='') as file:
    reader = csv.DictReader(file)
    for row in reader:
        if row['subject'] == '1' and row['test_type'] == 'production':
            if row['correct'] == '1':
                n_correct += 1
            total_trials += 1

accuracy = n_correct / total_trials
print(accuracy)

So, here, I'm manually isolating the rows where `subject == 1` and `test_type == production` and then calculating the accuracy. This is still not a great solution, however. I mean, it looks pretty messy, right?

A much better solution would be to use a full-fledged data frame to handle this kind of data. We won't look at data frames today – we'll return to this topic tomorrow – but to give you a quick preview, here's how you would do it using the `pandas` package. Pandas is not part of the Python Standard Library but it is included with Anaconda, so you may need to install it first to try out the following code. Installing packages is also something I will come back to later, so if you're not sure how to do it yet, don't worry, you can skip over this for now.

In [ ]:
import pandas as pd

# open the CSV file
df = pd.read_csv('data/example_data.csv')

# filter the rows of the data frame for subject==1 and test_type == production
subject_1 = df[ (df['subject'] == 1) & (df['test_type'] == 'production') ]

# calculate the accuracy
subject_1['correct'].mean()

For those of you coming from an R background, this will probably look very familiar. In any case, the main point is that, if you want to work with CSV files, it's probably best to use a package that's designed to work with tabular style data; the built-in CSV module is a bit too low-level for our typical use-cases. Whenever there are multiple options available to you, it's always a good idea to explore a few of them to see what's going to work best for your needs.

In [ ]:
######################### QUESTION 2 #########################
import quiz; quiz.display(quiz.HTML(quiz._unit3_questions[1]))
##############################################################

### JSON files

JSON stands for JavaScript Object Notation. Although the format originally came from the JavaScript world, it is now very widely used across many programming languages; there is nothing about it that is specific to JavaScript. JSON is a great format to use when you are working with data that is *not* naturally tabular. Let's jump right in and have a look at a JSON file from an experiment I ran recently:

In [ ]:
import json

with open('data/example_data.json') as file:
    data_set = json.load(file)
    
print(json.dumps(data_set, indent=4))

The `json` module is part of the Python Standard Library, so we don't need to install anything. Like the `csv` module, we need to open the file ourselves and then let the `json` module deal with parsing and reading the raw plain text. The resulting variable `data_set` is a Python dictionary containing all the data. In the last line I'm using the `json.dumps()` function to print the the dictionary with nice indentation to make it more human-readable, but this is not strictly necessary.

If you scroll through the data, you should be able to get a very rough idea of the design of the experiment. For example, the `trial_sequence` part lists all the trials that the participant did: first, a consent form, then a calibration, then some instructions, and then a training block, etc... And at the end of the file you will find the comments that the participant made and some information about what languages they speak.

A nice feature of JSON is that it is self-documenting. Because each value is accompanied by a key, it is relatively easy to understand what each piece of data means. The format is also very flexible in terms of the overall structure, allowing us to mirror the natural hierarchical structure of the experiment. It would be quite hard to force all this data into a table structure based on rows and columns. In which row and column would you put the participant's comments for example?

Since `data_set` is a Python dictionary, we can easily access the things that are of interest:

In [ ]:
print(data_set['comments'])

In [ ]:
print(data_set['other_languages'])

In [ ]:
print(data_set['creation_time'])
print(data_set['modified_time'])
print(data_set['modified_time'] - data_set['creation_time'])

Incidentally, the timestamps above are expressed in [UNIX time](https://en.wikipedia.org/wiki/Unix_time) – the number of seconds since 00:00:00 on 1st January 1970. This is a universal format – not just in Python – for representing dates and times without having to deal with all the awkwardness of non-decimal calendars and clocks. Here, `modified_time` is the last time the data was modified, so the difference between the two times tells us how long the participant took to complete the experiment (in seconds).

With a little bit of code, we can pull out the test trials and calculate accuracy:

In [ ]:
accuracy_by_position = [0, 0, 0, 0, 0, 0, 0]

for trial in data_set['responses']:
    if trial['test_type'] == 'controlled_fixation_test':
        correct = trial['object'] == trial['selected_object']
        accuracy_by_position[trial['fixation_position']] += correct

print(accuracy_by_position)

The details of the experiment don't matter, I mostly just want to give you the idea that you can work with JSON data quite naturally in Python and pull out the things that are interesting to you. JSON is also easy to use from the perspective of the experiment script; as the experiment is running, you simply store data in a Python dictionary, and then, at the end of the experiment, you write that dictionary to JSON – it's structure will be perfectly preserved.

That being said, it is often easier to work with numbers when they are arranged in a table – especially, when it comes to the statistical part of your analysis pipeline. It may, therefore, be useful to combine both formats. You could use JSON to store the raw data that comes directly out of your experiment, and then you could write a script to transform that raw data into a clean CSV file that can be passed into your statistical code. This CSV file would just contain the numbers relevant to the analyses; all the metadata, like timestamps and comments, would be left in the JSON in case you ever needed to refer back to it.

It might also be the case that neither CSV nor JSON is appropriate for your work. If you are doing some heavy computer simulations or working with large datasets, other formats might be more appropriate, especially in terms of efficiency. There might also be specific formats that are commonly used in your field

The main message I want to give you is that you should spend some time thinking about your data organization. Try to put yourself in the shoes of another scientist who wants to use your data in the future. Can they easily understand it and work with it to answer their own questions? Will it still be possible to open and process the data in ten years? Do you need to have access to specialist software to open the files? Will this software still exist in the future, and, if so, will it still run on computers of the future? These kinds of questions can be particularly challenging in the context of, for example, imaging data or eye tracking data, and there aren't necessarily good answers to these questions – you just have to do the best you can.

In [ ]:
######################### QUESTION 3 #########################
import quiz; quiz.display(quiz.HTML(quiz._unit3_questions[2]))
##############################################################

## Databases

Databases are a huge topic in computer science, but they are actually not used very often within academia, so I won't spend too much time covering them here. I just want you to know that databases exist and when you might need to use them.

The biggest feature that distinguishes databases from regular files (like CSV or JSON) is shared access. A database is a great choice when many different people or many different computational processes need to access a single shared dataset. Databases usually have some mechanism that enforces a **single source of truth** – the data stays in once place and only one person at a time can query or modify the data. This avoids synchronization problems (two people have different copies of the data) and data-race problems (two people try to modify the same piece of data at the same time).

Although these scenarios are really common in industry (imagine a bank with millions of customers), in science and academia we don't usually encounter these kinds of problems. Usually you'll be working directly with data files on your own computer, so there's not much need for a database. Moreover, databases can be quite complex to manage: often they require some kind of separate server process or they may even be hosted on a separate dedicated machine or in cloud storage.

However, if you're working in a team with lots of shared access to a single dataset, it may be worth investigating databases in more depth, particularly if everyone needs to be able to *write* to the database (not just read). There are many different databases that have different tradeoffs, and they can be broadly categorized into **relational databases** (e.g. [MySQL](https://en.wikipedia.org/wiki/MySQL) and [Postgres](https://en.wikipedia.org/wiki/PostgreSQL)) which have a tabular structure (similar to CSV) and so-called **noSQL databases** (e.g. [MongoDB](https://en.wikipedia.org/wiki/MongoDB) and [Redis](https://en.wikipedia.org/wiki/Redis)) that are more unstructured (similar to JSON).

If you want to try experimenting with databases, a good place to start is [SQLite](https://en.wikipedia.org/wiki/SQLite) – this is one of the simplest kinds of database that basically functions like a single file on your computer. Python has built-in support for SQLite databases (you can just `import sqlite3`). For other databases, you'll need to install the relevant Python package to manage the connection to the database.

## Pain with Paths

I'm sure all of you are very experienced with working with folders on your computer, and I'm sure all of you are super organized! Nevertheless, as your projects grow, things can quickly get out of control and messy. It is therefore also a good idea to think about not just the organization of your data, but also the organization of your entire project. Do you have data, manuscripts, stimuli, and code scattered all over the place with crazy long file names? Then it's time to get organized!

I'm not going to spend a lot of time here giving organizational advice; instead, I would suggest that you go read this website: https://goodresearch.dev (another one for the weekend). It is full of really excellent advice on how to keep your projects organized, particularly in the context of Python (although a lot of the advice is more general).

What I do want to focus on here is handling file paths. We have already used a very simple kind of path above. When opening a file, like `some_file.txt`, we are specifying the path to where the file can be found. In this case, since the file is in the same folder as the code (i.e. this notebook), we only need to specify the name of the file; the rest of the path is implied. However, as your project grows, you will need to organize data into subfolders and subsubfolders, so it's therefore really useful to understand how paths are handled in Python.

First, let's rewind a little and check that we're all on the same page in terms of what a path is. A full path is something like this:

```
/Users/jon/Code/sciprog26/some_file.txt
```

It describes where a file is located on your computer. When specifying paths in Python, you need to be careful to store the path as a string, otherwise Python will try to treat the `/` operator (divide) literally: It will try to calculate `Users` divided by `jon` divided by `Code` etc... After many years of programming, I still frequently forget to put paths inside quotation marks and then Python freaks out trying to divide a bunch of variables that don't exist. So, your path should be in quotation marks like this:

```python
"/Users/jon/Code/sciprog26/some_file.txt"
```

 For people on Windows, your paths will look a little different, maybe something like this:

```python
"C:\Users\jon\Code\sciprog26\some_file.txt"
```

In particular, note that Windows uses the backslash (`\`) as the path separator. If you're using Windows then you've probably discovered this issue already, since some of the code earlier in this notebook used forward slashes. Using backslashes in paths is a bit problematic because the backslash has a special meaning in Python and in many other languages – it is used as the ["escape character"](https://en.wikipedia.org/wiki/Escape_character). For example, `\n` represent a new line and `\t` represents a TAB character. Therefore, to specify paths on Windows you need to escape the escape character (!) by using double backslashes, so that you end up with something like this:

```python
"C:\\Users\\jon\\Code\\sciprog26\\some_file.txt"
```

An alternative solution is to use "raw strings" which are created similarly to the f-strings that we've already been using. In a raw string, which you use by placing an `r` in front the opening quotation mark, backslashes are not treated as special escape characters, allowing you to specify the path literally:

```python
r"C:\Users\jon\Code\sciprog26\some_file.txt"
```

If you're using a Mac or Linux, this shouldn't be an issue because paths are specified with forward slashes.

Anyway, to read in the file we created earlier, we can specify the full path:

In [ ]:
with open('/Users/jon/Code/sciprog26/some_file.txt') as file_object:
    content_of_the_file = file_object.read()
    
print(content_of_the_file)

Naturally, the code above won't work for you because the path is specific to my computer – it contains my username, for example. Go ahead and edit the path to make sure you can open the file correctly using its full path. You will need to figure out what the appropriate path is based on where you saved the file on your computer.

Hardcoding full paths like this is a bad idea, but, sadly, it's very common practice in science. What happens if a colleague tried to run your code? It won't work until they fix the path(s). And what happens if you move between Windows and Mac? For example, I code my experiments on my Mac but then I run them on a Windows computer in the lab, which can sometimes lead to annoying problems, like changing all the forwardslashes to backslashes.

There are two things you can do to avoid these kinds of issues, and it's a good idea to get into these two habits early on. First, you should always try to specify paths relative to the where the code is located. For example, let's say your project is organized like this:

```
project
|- code
|  |- analysis.py
|- data
|  |- exp1
|  |  |- data.csv
|  |- exp2
|  |  |- data.csv
```

When the Python script (`analysis.py`) is run, it is run from the `code` directory. Therefore, to access the data files, the code needs to move *up* into the `project` directory and then *down* into the data directory and then *down* again into, for example, the `exp1` directory. This is how we express the path from the perspective of the analysis script:

```python
"../data/exp1/data.csv"
```

or on Windows

```python
r"..\data\exp1\data.csv"
```

Note that the two dots (`..`) means "move up one directory." Thus, if someone downloads your entire project repository, they can run the `analysis.py` script and it will be able to locate `data.csv` with no issues. They don't need to fiddle around with your code, finding and changing all the paths. Unless, of course, this person is using a different operating system...

This brings me to the second thing you should always do. Instead, of writing paths as strings, use `Path` objects, which are designed specifically for representing paths. For example, to represent the path above, you would do this:

```python
Path('..') / 'data' / 'exp1' / 'data.csv'
```

Because we are using a `Path` object here, the forward slash `/`, which usually means divide, is automatically redefined to mean concatenate. (This might sound a bit crazy, but if you followed the bonus class on object-oriented programming, then you will recognize this as operator overloading.) Underlyingly, the path will automatically use the appropriate path separator depending on the OS you happen to be using. This allows you to specify paths in a OS-agnostic way.

To use the `Path` object, it needs to be imported from the `pathlib` module, as shown below. The `Path` object also allows us to do much more than simply *representing* paths. For example, it also allows you to iterate over all files in a path, for example:

In [ ]:
from pathlib import Path

home_dir = Path.home()
print(f'My home directory is {home_dir}')

print('These are the files in my home directory:')
for item in home_dir.iterdir():
    if item.is_file():
        print(item)
        
print('These are the folders in my home directory:')
for item in home_dir.iterdir():
    if item.is_dir():
        print(item)

This could be useful, for example, if you needed to iterate over all CSV files in a particular directory. Maybe, for example, you have a separate data file for each experiment you've done, and you need to iterate through them all to do some kind of processing. For example, here I am iterating over all the CSV files in the `data` directory alongside this notebook:

In [ ]:
data_directory = Path.cwd() / 'data'

for item in data_directory.iterdir():
    if item.suffix == '.csv':
        print(item)

`Path` objects have lots more handy methods and attributes for working with the file system. These include:

- `exists()` - Returns `True` if the path exists
- `mkdir()` - Create a directory on the file system
- `rename()` - Rename the file
- `parent` - The parent directory
- `stem` - The file name without the file name extension
- `suffix` - The file name extension

If you've ever needed to do batch renaming of 100s of files, you know how tedious it can be. The tools contained in `pathlib` should help you out. For a comprehensive review of all the things you can do, check out this link: https://realpython.com/python-pathlib/ My main pieces of advice are (1) to get into the habit of using `Path` objects and (2) to try to specify paths relatively rather than absolutely. Make sure that, if someone downloads your repository, they can just press run and immediately generate output without any fuss.

In [ ]:
######################### QUESTION 4 #########################
import quiz; quiz.display(quiz.HTML(quiz._unit3_questions[3]))
##############################################################

## List Comprehensions

Okay! That's enough data management. For the rest of this class, let's play around with some more interesting coding topics. First, a very elegant feature of Python is the **list comprehension**, which allows you to generate lists in an intuitive and concise way. List comprehensions are very ["pythonic"](https://www.askpython.com/python/examples/pythonic-way-of-coding). You don't *have* to use list comprehensions, but they are quite nice and widely used, so you are bound to encounter them pretty soon. Here's a list comprehension to generate the first ten square numbers:

In [ ]:
squares = [x**2 for x in range(1, 11)]
print(squares)

As you can see, this mixes together some of the syntax of a for-loop with some of the syntax of a list. This is how we would write the code if we were just using a regular for-loop:

In [ ]:
squares = []
for x in range(1, 11):
    squares.append(x**2)
print(squares)

We get exactly the same output, but with the list comprehension the code is shorter and more elegant. List comprehensions can even incorporate an if-statement. For example, let's make a list of squares that are also even:

In [ ]:
even_squares = [x**2 for x in range(1, 11) if x**2 % 2 == 0]
print(even_squares)

To get some practice, try rewriting the list comprehension above as an ordinary for-loop and if-statement:

Now try rewriting the following code as a list comprehension, and check that you get the same output:

In [ ]:
numbers = []

for x in range(100):
    if x % 2 == 0:
        numbers.append(x**2)
print(numbers)

You should notice that the list comprehension is basically just a reordering of all the syntax into a more compact form. There is also such a thing **dictionary comprehensions** too:

In [ ]:
squares = {x: x**2 for x in range(20)}
print(squares)

Does the syntax make sense? Try writing a dictionary comprehension where the keys are the numbers 1 through 26 and the values are the corresponding letter of the alphabet, `1:'A'`, `2:'B'`, etc... (hint: the `chr()` function might prove useful).

Overall, list and dictionary comprehensions provide a more elegant syntax for quickly generating simple lists and dictionaries. However, they should not be abused. It's actually possible to create really crazy list comprehensions that combine multiple loops and conditions, and the code can quickly become totally unreadable. Never forget that one of the most important things about your code is that it is readable – not just to others but also to your future self. **Readability is always more important than brevity, cleverness, or shaving milliseconds off the compute time.**

In [ ]:
######################### QUESTION 5 #########################
import quiz; quiz.display(quiz.HTML(quiz._unit3_questions[4]))
##############################################################

## Type Hinting

Python is a "[dynamically-typed language](https://en.wikipedia.org/wiki/Dynamic_programming_language)" (unlike many other languages which are "[statically typed](https://en.wikipedia.org/wiki/Static_program_analysis)"). This means that the type of a variable (integer, string, Boolean, etc.) is determined while the program is running. For example, if we write a line of code like:

In [ ]:
name = "Susanna"

we don't need to specifically tell Python that the variable `name` is a string. When Python reads this line of code, it dynamically infers that `name` is a string based on the context. This is unlike some other languages where you have to explicitly declare the type of a variable when you first create it. For example, in a language like Swift, you might write something like `let name: String = "Susanna"` – *let there be a variable called 'name' of type `String` that is equal to 'Susanna'*).

In Python, we can also freely redefine the `name` variable to be some other type, for example:

In [ ]:
name = 900

As you can see, Python doesn't care that `name` has suddenly changed into an integer. In other languages, this is forbidden; once a variable is declared to have a certain type, the type cannot be changed. In general, dynamically-typed languages – like Python, R, and JavaScript – are slower because the interpreter has to constantly check the types of all variables to allow for the possibility that they might change over time. In statically-typed languages – like C, Rust, or Swift – the program can allocate specific chunks of memory for each variable in advance, which makes the program much more efficient.

The main benefit of a dynamically-typed language, like Python, is that you, the programmer, can write code quickly and easily without thinking too much about the low-level details – you can focus your attention at the algorithmic-level, rather than the implementational-level. This makes it very easy to write quick prototypes and short scripts. However, the lack of explicit typing in Python also has some downsides. In particular, it forces you, the programmer, to infer and remember the type of each variable as you read and write the code. Explicit typing can also help you to catch bugs before running the program, saving you more time in the long-run.

This is where Python's "type hinting" comes in!

Type-hinting (also referred to as "type annotations") is a relatively new feature of Python that has become quite popular over the past five years or so. Type hints are exactly what they sound like: hints (indizi, allusioni) about the types of your variables. Type hints are completely optional in Python, but, these days, many projects use type-hinting, so it's good to have some awareness of it.

Essentially, when you create a variable, you can provide a hint about it's type, like this:

In [ ]:
name: str = "Susanna"

This is read as "the variable 'name' of type string is equal to Susanna". You can also provide type hints in a function declaration, like so:

In [ ]:
def number_of_letters_in_name(name: str) -> int:
    n_letters: int = len(name)
    return n_letters

So the `number_of_letters_in_name()` function takes one argument called `name`, which should be a string, and the function returns an integer (`-> int`). Within the function itself, we created a variable `n_letters` of type int. All three of these hints are optional and none of them are strictly enforced. Personally, I don't usually bother with type hints for every variable – I find it overkill – but I do find it useful to put type hints in function declarations. If I come back to some code that I haven't looked at in a while, it's useful to have reminders about what types a function expects and returns. Another major benefit of type hints is that, in some code editors and IDE's, the software can provide helpful autocompletions and it can warn you when your code contains type mismatches (before you even run the code). This can really make programming feel so much easier.

If you want to learn a little more and see some of these things in action, check out [this YouTube video](https://www.youtube.com/watch?v=15WB30NqDT0).

In [ ]:
######################### QUESTION 6 #########################
import quiz; quiz.display(quiz.HTML(quiz._unit3_questions[5]))
##############################################################

## Error Handling

Over the past few days, I'm sure you've bumped into lots of errors, and maybe you sometimes feel frustrated with them. But one thing I want you to remember is that **errors are a good thing!** Errors inform you that something fishy is going on. It is much better to be aware of a potential problem than for the problem to go silently undetected.

### Exception handling with `try`/`except`

Sometimes, if you expect a particular type of error to occur, you might be able to take corrective action. Exception handling using the `try` and `except` statements can do exactly this. Here's an example:

In [ ]:
# pick a number...
number = 0

colors = {1:'red', 2:'green', 3:'blue'}

try:
    picked_color = colors[number]
except KeyError:
    picked_color = 'black'
    
print(picked_color)

Python will attempt to run the code inside the `try` block. If that code runs fine with no errors, then we just continue as normal. However, if that code fails, specifically if it produces a `KeyError`, the code in the `try` block is abandoned and the code inside the `except` block will run instead. In this case, if you pick the numbers `1`, `2`, or `3`, the code will run fine and you will get the corresponding color. If you choose some other number that is not a valid key in the `colors` dictionary, the code will fail with a `KeyError`, causing the variable `picked_color` to be set to `black`. Exception handling is especially useful if you expect a particular type of error to occur and you know in advance how to correct it.

### Raising errors with `raise`

The inverse of exception handling is explicitly forcing errors to occur. This is useful because it alerts you to a particular problem. To raise an error, you use the `raise` statement. For example, here we will perform a check to see if `number` is set to `1`, `2`, or `3` and raise a `ValueError` if it is not. As you can see we can also write a custom error message.

In [ ]:
# pick a number...
number = 0

if number not in [1, 2, 3]:
    raise ValueError('You must pick either 1, 2, or 3!')

colors = {1:'red', 2:'green', 3:'blue'}
picked_color = colors[number]
print(picked_color)

At first, it might seem odd to raise errors and crash your program – why would I ever want my code to crash!? But errors are actually really useful and can help you feel more confident about your code. If the program runs without crashing, you can feel more confident that things are working correctly. Explicit error messages can also help you quickly identify problems by providing a message that is specific to your particular program. For example, try removing the `if number not in [1, 2, 3]` condition and the raise statement. What happens when you run the code with an invalid number? Which error is more helpful?

### Asserting things to be `True` with `assert`

Another way to approach this is to write assertions with the `assert` statement. This allows you to assert that something is `True`, and if it turns out to be `False`, the code will crash.

In [ ]:
# pick a number...
number = 0

assert number in [1, 2, 3]

colors = {1:'red', 2:'green', 3:'blue'}
picked_color = colors[number]
print(picked_color)

A very smart thing to do is to put lots of `assert` statements all over your code. Make lots of obvious assertions – things that should obviously be `True`; one day one of those assertions will be be `False` forcing you to realize that something dodgy is going on. For example, in a script I wrote recently, I needed to pair up experimental trial data stored in one file with the corresponding eye tracker recording data in another file. I put in some `assert` statements which assert that the trial ID in the response data should match the trial ID in the eye tracker recording. In principle, everything should be fine and the two data sources should line up correctly. However, the assert statements give me extra peace of mind; if there's ever a mismatch for some obscure, unpredictable reason, I will see big red errors.

In [ ]:
######################### QUESTION 7 #########################
import quiz; quiz.display(quiz.HTML(quiz._unit3_questions[6]))
##############################################################

## Mini-Project

To finish the day, let's turn to a fun mini-project! Here's an [ASCII-art](https://en.wikipedia.org/wiki/ASCII_art) picture of a dog that I made with a small Python program (12 lines of code).

In [ ]:
'''
                                                                                                    
                                                                            `;I!!I:`                
                                                                      'I+](rrffjjjjjxt]}}+,         
   .                                                               ;/jrj)}->lIIIIII!++-+]/ujI       
  tzx+                                                            Imr>!IIlll!!!!!!l!C]IllIl]Y/.     
  Ot?Xu.                                                          ch{l!!!!!lIl!!!!l!m[l!!!!liuU<    
  {L<<cv                                                         /v<l!!!!!l[j[lll!IfL!!!!!!!lI]Jn`  
  .Q(<<vc'                                                       0u)l!!!!!!f/t!iil~m-l!!!!!!!!l!rQl 
   <Z-~<nU;                                                      Cmtl!!!!!!III<[~IUuI!!!!!!!!!!lIzQ'
    /C~~</U(`                                                 '1Uz}i!!!!!ll!!!i!I}mil!!!!!!!!!!!l1k,
     XY<~~+rU{`                                              iQkmOxl!!!!i?i!!!llILjI!!!!!!!!!!!!lcO`
     'YU+~~<?xz/i'                                          IaZCbhZ!l!!!!<i!!!!l~p+l!!!!!!!!!!!l/m: 
       x01~~~>i}jxf}<,                                      .(aqaXii~!<i<il!l!!l[m!l!!!!!!!!!!l+m-  
        ]ZY]!lllI!~{fxxf}<;'                                  Zt?~<!>!l~>iIl<!ll]m!l!!!!!!!!lI<Lj   
         ^rOz}iIllllIl!~])frr/?!`                             }YxqXXn1++<<[cui<>+q}Il!!!!llI!10j    
           ,)YYr}~!lIllllIl!>-{trf{~I:,:Ili~?][]??]]??][}]??<i~JZkLttuUZmwQ)i~++<nq[!lll!<[rJz<     
              ;}nYXn/[<!IllllllIl>]/jjjrjft({}[?-?][[[}}}}}}}{}~!~jYXXvf{1xL/~+++~r0LYXYCLmqI       
                 'I+)xzzn)+!Il!!!llIIllllIIIIlllllllllllllllllllllIl!!lIIII~]+++++<-(rrt1?/q;       
                       :+/cXfil!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!l!!!!!i>~++++++~~~~ill-Lz       
                           tm>l!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!>++++++++++?++~~i!/pi      
                          1U<l!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!<+++++++++~[(+++>l~Cz      
                         <Qil!!!!!ii!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!i~~><+++++++++++[(-++>!!td:     
                        .J[I!!!!!!<~!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!~++++++++++++++~[1~++++il-w1     
                        {UI!!!!!!!>+>!!!!!!!!!!!!!!!!!!!!!!!!!!!!l>+++++++++++++++~-/-~+++~>~Lu     
                        L{l!!!!!!li++~>!!!!!!!!!!!!!!!!!!!!!!!!!!!~+++++++++++++++~{}~++++++<Jz     
                        0]l!!!!!!!<++++i!!!!!!!!!!!!!!!!!!!!!!!!!i+++++++++++++++~}/+++++++~+Z/     
                        rnI!!!!!!<+++++~i!ll!!!!!!!!!!!!!!!!!!!!!>+++++++++++++++[]?~++++++~{q;     
                        {XI!!!!!i++++++++~>ii!!!!ll!!!!!!!!!!!!!!<+++++++++++++++{+~+++++++<zz      
                        YfI!ll!!<++++~<~~~++~~~~<iii!!!!!!!ll!!!!<+++++++++++++++~++++++++<{pI      
                        O[l!i><~+++~~[vCUt-~<<~~++++~<>!!>~>i!!i>~+++++++++++++++++++++++~[p)       
                        jnI>+++++++</pqmmmQCzx)?+~<<~~~<>i<~~<<+?+++++++++++++++++++++++~+0n        
                        lLi~++++++~fbx}1(ruzpvnYYcunrrxnnuvunx//0-~++++++++++++++~+++++~>v0.        
                        I0<~+++++~)kr>~~+tUZ{  'I<-][?--+<!!>?(Yd-~+++++~~~~~~<<~]-+++<l[d<         
                        1J<~~++~<}wz<+~?uZj;                   ,Z-~~+++++-?[)fjrCL?~++<<LX          
                       .L/~~~+~[vOv<+++cm<                     .0{~~~+~+cQXzr)}ZX?~++~~nm:          
                       nz<+++~twh1!~+~jml                       tU<~++~}q1'    X1<+~~-uwi           
                      {Q>~++~(p]fX>~~]m~                        `O/<~~~{q[     X(~+~~xq<            
                     .Z{~++~{wt !m<~<cY                          <m?<~~[Zx    .Q{~+~1p}             
                      Jr<+~?Qc  `m]~?q}                           1L~~<+vZ'   ,O?<~?UQ              
                      [Q<~~vO'   Xx>)dI                            cv<+~]Ot   ~Q~~~/d]              
                      -O~~{d~    -O~)p?>I.                         ^0(<+~(q>  (u<~+cm'              
                      fU<~YU     .Z]~{fnvC~                         ~O~~~~v0. zf~~]Qz               
                      Ct<~Z/      cYn(jwQQ/                          C/<~~]p~ vx<~[p]               
                     `m[~~{Ur;     ;-tj/i.                           [L<++~L/ nr<~?m{;'             
                     `m[<{j+fO~                                      'Z1~+~zY cn<~~?)rcj,           
                      )C1]CO/OY                                       jz<+~jw !Ot<<ft<?Xm`          
                       ijcYO/}:                                       [C<+~?Lt`>pf)rkQXv{.          
                          '                                           ?L<~+??JZ!<)/)}I.             
                                                                      lO}}>CjC#x                    
                                                                       fmZ/bQ0u^                    
                                                                        ,[r)-I                      
                                                                                                               
'''

Try building your own Python script that can convert any image into ASCII art. Feel free to work on the project with the people around you. Here are some tips:

1. How will you open and read the image file? There are some Python packages that can help you such as [Pillow](https://pillow.readthedocs.io/en/stable/handbook/tutorial.html).
2. You can use the following list of characters, which are ordered from "dark" characters (@B%) to "light" characters ('. ): @B%8&WM#oahkbdpqwmZO0QLCJUYXzcvunxrjft/()1{}[]?-+~<>i!lI;:,^`'.
3. Somehow you want to map each pixel to a character, with darker pixels mapping to darker characters.
4. You might need to do some resizing of the image to get it to look right.

## More...

Still want more...? Check out this page of 100 tips and tricks in Python: https://holypython.com/100-python-tips-tricks/ Peruse the list and try out the ones that seem interesting to you.

Spend some time thinking about the organization of your current projects. Do you need to do some housekeeping? Is your project well documented? Is your data open and accessible? Are the results reproducible with minimal effort? Check out https://goodresearch.dev/ and https://tidyr.tidyverse.org/articles/tidy-data.html for solid advice.